## Exercises on Actor–Critic Methods

These paper-and-pencil exercises reinforce Chapter 10: the advantage function and its centring property, the $n$-step actor–critic target (with the **correct** baseline $V(S_t)$), the TD error as a one-step advantage, Generalised Advantage Estimation (GAE), the bias–variance trade-off between Monte Carlo and bootstrapped critics, and entropy regularisation. They build on the policy-gradient foundations of Chapter 09. Notation: actor parameters $\theta$, critic $V(\cdot;\phi)$, advantage $A(s,a)$, TD error $\delta_t$.

### Exercise 10.1 — The advantage function and its centring property

In a state $s$ the action-values are $q_\pi(s,\cdot)=(2,\,0,\,1)$ for three actions, and the policy is $\pi(\cdot\mid s)=(0.5,\,0.3,\,0.2)$.

1. Compute the state value $v_\pi(s)$ and the advantages $A_\pi(s,a)=q_\pi(s,a)-v_\pi(s)$.
2. Verify that the policy-weighted advantage is zero, and explain why this matters.

**Step 1 — State value and advantages.** $\ v_\pi(s)=\sum_a\pi(a\mid s)\,q_\pi(s,a) = 0.5(2)+0.3(0)+0.2(1)=1.2.$

$\displaystyle A_\pi(s,\cdot) = q_\pi(s,\cdot)-v_\pi(s) = (2,0,1)-1.2 = (0.8, -1.2, -0.2).$

**Step 2 — Centring.** $\ \sum_a\pi(a\mid s)\,A_\pi(s,a) = 0.5(0.8)+0.3(-1.2)+0.2(-0.2) = 0.0.$

This is no accident: $\sum_a\pi A = \sum_a\pi q - v\sum_a\pi = v - v = 0$. Under the policy, advantages average to zero — positive for better-than-average actions, negative for worse.

**Key concept**

The advantage measures *how much better than the policy's average* an action is. Because it is centred at zero, using it as the weight in the policy gradient increases the probability of above-average actions and decreases below-average ones, giving a lower-variance signal than the raw return.

### Exercise 10.2 — The $n$-step actor–critic target

From state $S_t$ the agent collects a 3-step segment with rewards $R_{t+1},R_{t+2},R_{t+3}=0,0,1$ and then bootstraps on the critic. With $\gamma=0.9$, critic estimates $V(S_t)=0.5$ and $V(S_{t+3})=0.7$:

1. Compute the 3-step return $G_{t:t+3}$.
2. Compute the $n$-step advantage $A(S_t,A_t)$ and state the critic's regression target.

**Step 1 — $n$-step return** (three real rewards, then bootstrap on $V(S_{t+3})$):

$\displaystyle G_{t:t+3} = R_{t+1}+\gamma R_{t+2}+\gamma^2 R_{t+3}+\gamma^3 V(S_{t+3}) = 0+0+0.81(1)+0.729(0.7) = 1.3203.$

**Step 2 — Advantage and critic target.** The advantage subtracts the value of the **current** state $S_t$ (the baseline is $V(S_t)$, *not* $V(S_{t+n})$):

$\displaystyle A(S_t,A_t) = G_{t:t+3} - V(S_t) = 1.3203 - 0.5 = 0.8203.$

The **actor** uses $A(S_t,A_t)$ to weight $\nabla_\theta\log\pi_\theta(A_t\mid S_t)$; the **critic** regresses $V(S_t)$ toward the target $G_{t:t+3}=1.3203$ (loss $\tfrac12(G_{t:t+3}-V(S_t))^2$).

**Key concept**

An actor–critic replaces the Monte Carlo return with an $n$-step **bootstrapped** return: fewer real rewards plus the critic's estimate of the rest. The bootstrap introduces bias (the critic is imperfect) but cuts variance. The baseline subtracted is always the value of the state where the action was taken, $V(S_t)$.

### Exercise 10.3 — Generalised Advantage Estimation (GAE)

Along a trajectory the one-step TD errors are $\delta_t=R_{t+1}+\gamma V(S_{t+1})-V(S_t)$, computed as $(\delta_0,\delta_1,\delta_2)=(0.2,\,0.5,\,1.0)$. With $\gamma=0.9$ and $\lambda=0.5$:

1. Explain why $\delta_t$ is itself a (one-step) advantage estimate.
2. Compute the GAE advantage $A^{\text{GAE}}_0 = \sum_{l\ge0}(\gamma\lambda)^l\,\delta_{t+l}$ at $t=0$.
3. Give the $\lambda=0$ and $\lambda=1$ limits.

**Step 1 — $\delta_t$ as an advantage.** $\ \delta_t = R_{t+1}+\gamma V(S_{t+1}) - V(S_t)$ is the one-step return minus the baseline $V(S_t)$ — exactly a one-step estimate of $A(S_t,A_t)$ (an unbiased estimate of the advantage when $V=v_\pi$).

**Step 2 — GAE at $t=0$** with discount $\gamma\lambda=0.9(0.5)=0.45$:

$\displaystyle A^{\text{GAE}}_0 = \delta_0 + (\gamma\lambda)\delta_1 + (\gamma\lambda)^2\delta_2 = 0.2 + 0.45(0.5) + 0.2025(1.0) = 0.6275.$

**Step 3 — Limits.**
- $\lambda=0$: $\ A^{\text{GAE}}_0 = \delta_0 = 0.2$ — the **one-step** (high-bias, low-variance) advantage.
- $\lambda=1$: $\ A^{\text{GAE}}_0 = \sum_l \gamma^l\delta_l = 0.2+0.9(0.5)+0.81(1.0) = 1.46$ — the **Monte Carlo** (low-bias, high-variance) advantage.

**Key concept**

GAE is the advantage analogue of the $\lambda$-return: an exponentially weighted average of $n$-step advantage estimates, controlled by $\lambda$. It lets us dial continuously between the low-variance one-step critic ($\lambda=0$) and the unbiased Monte Carlo estimate ($\lambda=1$), typically choosing $\lambda\approx0.95$.

### Exercise 10.4 — Bias and variance of critic targets (integrative)

From state $s$ the dynamics are: a deterministic step with reward $R_{t+1}=0$ into $s_1$, then from $s_1$ a reward $R_{t+2}$ that is $+2$ or $0$ with equal probability (mean $1$), then termination; $\gamma=1$. The true values are $v(s_1)=1$ and $v(s)=1$.

1. Compute the mean and variance of the **Monte Carlo** target $G_t$ used to update $V(s)$.
2. Compute the mean and variance of the **one-step (bootstrapped)** target $R_{t+1}+\gamma V(s_1)$, first with an exact critic $V(s_1)=1$, then with a biased critic $V(s_1)=0.8$.
3. Summarise the bias–variance trade-off.

**Step 1 — Monte Carlo target.** $\ G_t = R_{t+1}+R_{t+2} = 0 + R_{t+2} \in\{0,2\}$, each with prob. $0.5$.

$\displaystyle \mathbb{E}[G_t] = 1 \ (\text{unbiased, since }v(s)=1), \qquad \mathrm{Var}[G_t] = \tfrac12(0^2+2^2)-1^2 = 2-1 = 1.$

**Step 2 — One-step bootstrapped target** $R_{t+1}+\gamma V(s_1) = 0 + V(s_1)$ (a *constant*, since $R_{t+1}$ is deterministic and $V(s_1)$ is fixed):

- Exact critic $V(s_1)=1$: target $=1$, $\ \mathbb{E}=1$ (unbiased), $\ \mathrm{Var}=0$.
- Biased critic $V(s_1)=0.8$: target $=0.8$, $\ \mathbb{E}=0.8$ (**bias** $=-0.2$), $\ \mathrm{Var}=0$.

**Step 3 — Trade-off.** The MC target is **unbiased** but carries the full variance of the stochastic future reward ($\mathrm{Var}=1$). The bootstrapped target has **zero variance** here (it replaces the random future by the critic's number) but is **only as correct as the critic**: an imperfect $V(s_1)$ injects bias ($-0.2$). Actor–critic methods accept a little bias to buy a large variance reduction; GAE (Exercise 10.3) tunes exactly where on this spectrum to sit.

**Key concept**

Bootstrapping trades variance for bias. Monte Carlo returns are unbiased but noisy; one-step critic targets are stable but biased by critic error. The whole design space of actor–critic methods — $n$-step returns, GAE, target networks — is about managing this trade-off.

### Exercise 10.5 — Entropy regularisation

For a two-action policy with $\pi=(p,\,1-p)$, the entropy is $H(\pi)=-\big[p\ln p + (1-p)\ln(1-p)\big]$ (in nats).

1. Compute $H$ for the uniform policy $p=0.5$ and for the near-deterministic policy $p=0.9$.
2. Show that entropy is maximised at $p=0.5$, and explain how adding $\beta H(\pi)$ to the actor objective affects learning.

**Step 1 — Evaluate.**

$\displaystyle H(0.5,0.5) = -\big[0.5\ln 0.5 + 0.5\ln 0.5\big] = \ln 2 = 0.6931\ \text{nats},$
$\displaystyle H(0.9,0.1) = -\big[0.9\ln 0.9 + 0.1\ln 0.1\big] = 0.3251\ \text{nats}.$

The near-deterministic policy has **lower** entropy.

**Step 2 — Maximum at uniform.** Setting $\tfrac{dH}{dp} = -\ln p + \ln(1-p) = \ln\tfrac{1-p}{p} = 0$ gives $p=0.5$; the second derivative $-\tfrac1p-\tfrac1{1-p}<0$ confirms a maximum. So entropy is largest for the uniform policy and $0$ for a deterministic one.

**Step 3 — Effect of the bonus.** Maximising $J(\theta)+\beta H(\pi_\theta)$ rewards *keeping the policy spread out*. Early in training this **discourages premature collapse** onto one action (which would set $H\to0$ and stop exploration); as learning proceeds, the return term dominates and the policy is allowed to sharpen. $\beta$ controls the strength of this pressure.

**Key concept**

An entropy bonus is a built-in exploration incentive for policy-gradient methods: it penalises over-confident policies, keeping action probabilities diverse long enough to discover good actions — the on-policy counterpart of $\varepsilon$-greedy / optimistic exploration.